<a href="https://colab.research.google.com/github/danimb-git/cardio-disease-ml-uab/blob/main/notebooks/Main_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting the presence of cardiovascular disease
**Course:** Fundamentals of Machine Learning  
**Project team:** Team 8 - Ariadna López Traver, Daniel Moore Burgos, Alejandro Ucelayeta Jarque  
**Dataset:** [Cardiovascular Disease Dataset (Kaggle)](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset?resource=download$0)

## 1. Project Objective & Problem Formulation
Cardiovascular diseases (CVDs) are a group of conditions affecting the heart and blood vessels, and are one of the leading causes of mortality globally.

The objective of this project is to build, evaluate, and analyze supervised binary classification models capable of predicting the presence or absence of CVDs from clinical examination data, patient metrics, and lifestyle habits.

### Dataset features

The dataset consists of 70,000 patient records described by 11 input features (such as age, gender, systolic blood pressure, diastolic blood pressure, etc) and 1 target variable. The target class "cardio" equals to 1 when patient has cardiovascular desease, and 0 if patient is healthy.

There are 3 types of input features:
* Objective: factual information.
* Examination: results of medical examination.
* Subjective: information given by the patient.

| Feature | Variable Type | Variable | Value Type | Description |
| :--- | :--- | :--- | :--- | :--- |
| Age | Objective Feature | `age` | int (days) | Patient's age in days |
| Height | Objective Feature | `height` | int (cm) | Patient's height |
| Weight | Objective Feature | `weight` | float (kg) | Patient's weight |
| Gender | Objective Feature | `gender` | categorical code | 1: women, 2: men |
| Systolic blood pressure | Examination Feature | `ap_hi` | int (mmHg) | Systolic blood pressure |
| Diastolic blood pressure | Examination Feature | `ap_lo` | int (mmHg) | Diastolic blood pressure |
| Cholesterol | Examination Feature | `cholesterol` | ordinal (1, 2, 3) | 1: normal, 2: above normal, 3: well above normal |
| Glucose | Examination Feature | `gluc` | ordinal (1, 2, 3) | 1: normal, 2: above normal, 3: well above normal |
| Smoking | Subjective Feature | `smoke` | binary | Self-reported smoking status |
| Alcohol intake | Subjective Feature | `alco` | binary | Self-reported alcohol intake |
| Physical activity | Subjective Feature | `active` | binary | Self-reported physical activity |
| Cardio (Target) | Target Variable | `cardio` | binary | Presence (1) or absence (0) of cardiovascular disease |

### Methodology and team workflow

To ensure methodological rigor, we follow the OSEMN workflow (Obtain, Scrub, Explore, Model, iNterpret).

The team uses an anti-leakage protocol: the dataset is split into training (80%) and test (20%) sets immediately after basic structural verification. All exploratory data analysis (EDA), outlier handling, feature engineering, scaling, and cross-validation will be performed exclusively on the training set.

A fixed global `RANDOM_SEED` is enforced across all splits and models to ensure reproducibility.

The team manages the project through a shared GitHub repository structured into *data/* (hosting the raw dataset) and *notebooks/* (hosting the master notebook, *Main_Pipeline.ipynb*). Using Google Colab integrated with GitHub allows all three members to run and update the notebook consistently from any machine, while a portable data loader (checking local paths first with the GitHub raw data fallback) ensures seamless execution.

## 2. Environment Setup and Reproducibility

In this section we import the main scientific computing and visualization libraries we will use throughout the notebook, define a global random seed to ensure reproducibility of data splits, and log the exact library versions used in the project.

In [505]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split

In [506]:
# Gemini recomends supresing non-critical warnings to keep the notebook outputs clean.
# import warnings
# warnings.filterwarnings('ignore')

In [507]:
# Definition of global random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [508]:
# Gemini recommends setting default values for consistent visual styling for the
# exploratory data analysis and for report figures
# sns.set_theme(style="whitegrid", context="notebook", palette="muted")
# plt.rcParams["figure.figsize"] = (10, 6)
# plt.rcParams["axes.titlesize"] = 13
# plt.rcParams["axes.labelsize"] = 11
# Display all columns when inspecting DataFrames
# pd.set_option("display.max_columns", None)
# pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [509]:
# Library versions
print("Python version :", sys.version.split()[0])
print("NumPy version :", np.__version__)
print("Pandas version :", pd.__version__)
print("Scikit-learn version :", sklearn.__version__)
print("Global RANDOM_SEED :", RANDOM_SEED)

Python version : 3.13.18
NumPy version : 2.4.6
Pandas version : 2.2.3
Scikit-learn version : 1.6.1
Global RANDOM_SEED : 42


## 3. Data acquisition and structural audit

Before splitting the data into the training and test sets, we perform structural checks of the dataset (dimensions, data types, duplicates, etc.).
1. Load the CSV file (Kaggle files use ";" as delimiter).
2. Dataset dimensions, column names, and data types.
3. Check if the dataset contains missing values (NaN).
4. Remove the `id` column. It is an arbitrary identifier with no predictive meaning (models should not have the id to learn from it).
5. Check for duplicate rows (exact duplicates excluding `id`). Leaving identical records across both the training and test sets would inflate test metrics.

In [510]:
# We try different local paths to ensure it will work in the following scenarios:
#  - "data/cardio_train.csv": running locally from the project root.
#  - "../data/cardio_train.csv": running locally from inside the 'notebooks/' folder (our repository stucture).
#  - "cardio_train.csv": running locally with the CSV in the same folder as the notebook.
# If non of the local paths work, we use the GitHub url
local_paths = ["data/cardio_train.csv",
               "../data/cardio_train.csv",
               "cardio_train.csv"]
github_url = "https://raw.githubusercontent.com/danimb-git/cardio-disease-ml-uab/main/data/cardio_train.csv"

for path in local_paths:
  if os.path.exists(path):
    file = path
    print("Loading dataset from local path: ", path)
    break
else:
  file = github_url
  print("Loading dataset from GitHub Raw URL")

df_raw = pd.read_csv(file, sep=';')

print("Raw dataset shape:", df_raw.shape[0], "rows x", df_raw.shape[1], "columns")
display(df_raw.head())

Loading dataset from local path:  cardio_train.csv
Raw dataset shape: 70000 rows x 13 columns


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [511]:
# Dataset dimensions, column names, and data types
print("Dataset info:")
df_raw.info()

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [512]:
# Check if the dataset contains missing values (NaN)
print("Explicit missing values per column:")
missing_counts = df_raw.isna().sum()
if missing_counts.sum() > 0:
  print(" ", missing_counts[missing_counts > 0])
else:
  print(" No explicit missing values found.")

Explicit missing values per column:
 No explicit missing values found.


In [513]:
# Remove the id column
df = df_raw.drop(columns=['id']).copy()
display(df.head())

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [514]:
# Check for duplicate rows
n_duplicates = df.duplicated().sum()
print("Number of duplicate rows: ", n_duplicates)
if n_duplicates > 0:
  print("Sample of duplicate rows:")
  # .duplicates(keep=False) to keep all duplicated rows (including the first one)
  # .sort_values(by=list(df.columns)) to display duplicated rows together
  display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head())
# Remove duplicate rows
# .reset_index(drop=True) to reconstruct indexes from 0 to the new length of the DataFrame
df_deduplication = df.drop_duplicates().reset_index(drop=True)
print("\nDataset shape after removing duplicates: ", df.shape[0], "rows x", df.shape[1], "columns")

Number of duplicate rows:  24
Sample of duplicate rows:


,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
6325,14552,1,158,64.0,120,80,1,1,0,0,1,0
40365,14552,1,158,64.0,120,80,1,1,0,0,1,0
17101,16160,1,168,65.0,120,80,1,1,0,0,1,1
64169,16160,1,168,65.0,120,80,1,1,0,0,1,1
1204,16793,1,165,68.0,120,80,1,1,0,0,1,0



Dataset shape after removing duplicates:  70000 rows x 12 columns


## 4. Stratified train and test split

We isolate 20% of the data as a test set before performing any exploratory analysis, outlier cleaning, or preprocessing.

Because physiological measurements (height and weight) and cardiovascular risk profiles differ by the gender of the patient, we decided to perform stratified sampling using a composite key combining `cardio` and `gender`.
This composite stratification ensures that both the target class and the gender distributions are preserved across the training and test sets.

From this point forward, all EDA, cleaning rules, feature engineering, model selection, and cross-validation will be performed using only the training set (`X_train` and `y_train`). `X_test` and `y_test` will be reserved and untouched until the final model evaluation.

In [515]:
# Separate predictors and target
X = df_deduplication.drop(columns=['cardio'])
y = df_deduplication['cardio']

In [516]:
# Create composite stratification key
# 'cardio' + 'gender' --> ("0_1", "0_2", "1_1" and "1_2")
stratification_key = y.astype(str) + "_" + X['gender'].astype(str)
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=stratification_key
)

In [517]:
# Reset indices from new sets
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [518]:
# Assemble training DataFrame
train_df = X_train.copy()
train_df['cardio'] = y_train

In [519]:
print("'train_df'' shape:", train_df.shape[0], "rows x", train_df.shape[1], "columns")
print("'X_train' shape:", X_train.shape[0], "rows x", X_train.shape[1], "columns")
print("'y_train' shape:", y_train.shape[0], "rows")
print("'X_test' shape:", X_test.shape[0], "rows x", X_test.shape[1], "columns")
print("'y_test' shape:", y_test.shape[0], "rows")

'train_df'' shape: 55980 rows x 12 columns
'X_train' shape: 55980 rows x 11 columns
'y_train' shape: 55980 rows
'X_test' shape: 13996 rows x 11 columns
'y_test' shape: 13996 rows


In [520]:
# Verify stratification
print("'Cardio' = 1 proportion in training set: ", round(100 - y_train.mean() * 100,2), "%")
print("'Cardio' = 1 proportion in test set: ", round(100 - y_test.mean() * 100,2), "%")
print("'Cardio' = 0 proportion in training set: ", round(y_train.mean() * 100,2), "%")
print("'Cardio' = 0 proportion in test set: ", round(y_test.mean() * 100,2), "%")
print("\n'Gender' = 1 proportion (women) in training set: ", round((X_train['gender'] == 1 ).mean() * 100, 2), "%")
print("'Gender' = 1 proportion (women) in test set: ", round((X_test['gender'] == 1 ).mean() * 100, 2), "%")
print("'Gender' = 2 proportion (men) in training set: ", round((X_train['gender'] == 2 ).mean() * 100, 2), "%")
print("'Gender' = 2 proportion (men) in test set: ", round((X_test['gender'] == 2 ).mean() * 100, 2), "%")

'Cardio' = 1 proportion in training set:  50.02 %
'Cardio' = 1 proportion in test set:  50.02 %
'Cardio' = 0 proportion in training set:  49.98 %
'Cardio' = 0 proportion in test set:  49.98 %

'Gender' = 1 proportion (women) in training set:  65.04 %
'Gender' = 1 proportion (women) in test set:  65.03 %
'Gender' = 2 proportion (men) in training set:  34.96 %
'Gender' = 2 proportion (men) in test set:  34.97 %
